# NB04 — BTCS Multi-Dataset Evaluation + Baselines
**Andre da Costa Silva | ITA | 2026**

Pipeline genérica: BTCS v3 + 4 baselines em múltiplos datasets.

| Dataset | Tipo label | Edges | Temporal |
|---------|-----------|-------|----------|
| AML100k / AML1M | edge (GNN scores) | 2.49M / 24M | sim |
| Bitcoin OTC / Alpha | edge (ratings) | 35k / 24k | sim |
| IBM AML txns | edge (IS_FRAUD) | 1.3M | sim |
| IBM HI-Small / Medium / Large | edge (Is Laundering) | 5M / 32M / 180M | sim |
| IBM LI-Small / Medium / Large | edge (Is Laundering) | 7M / 31M / 176M | sim |
| Elliptic | node → edge | 234k | sim |
| Amazon / Yelp Fraud | node → edge (.mat) | 4.4M / 3.8M | não |
| PaySim | txn-level | 6.3M | sim |

**Métodos:**
- **BTCS v3**: WCC → Leiden hierárquico com janela temporal + budget cap
- **B0 Random**: Partição aleatória dos top-k edges
- **B1 WCC-only**: Apenas WCC, sem Leiden split
- **B2 Louvain**: Community detection (Louvain) no subgrafo top-k
- **B3 Greedy**: BFS seed expansion com budget constraint

**Métricas**: Coverage, Purity, OCR(B), n_cases, time

Executar: Runtime → Run all (A100 recomendado para IBM Medium/Large)

In [ ]:
# CELULA 1 - Setup
import os, sys, subprocess, time, math, contextlib, warnings
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

_pip('numpy', 'pandas', 'scikit-learn', 'matplotlib', 'psutil',
     'python-igraph', 'leidenalg', 'scipy')
try:
    import torch
except ImportError:
    _pip('torch', '--index-url', 'https://download.pytorch.org/whl/cu121')
    import torch

import numpy as np
import pandas as pd
import psutil
import igraph as ig
import leidenalg
import matplotlib.pyplot as plt
from IPython.display import display

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

BASE = Path('/content/drive/MyDrive') if IN_COLAB else Path('.').resolve()
DATA = BASE / 'GrafosGNN/data'
OUT  = BASE / 'GrafosGNN/results/nb04_multi_dataset'
OUT.mkdir(parents=True, exist_ok=True)

# Paths AML datasets (pre-treinados)
AML100K_BASE  = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML100k'
AML1M_BASE    = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML1M'

print(f'Base: {BASE}')
print(f'Data: {DATA}')
print(f'Out:  {OUT}')

# Verificar datasets disponíveis
print('\n=== Datasets disponíveis ===')
for name in ['bitcoin_otc', 'bitcoin_alpha', 'paysim', 'elliptic',
             'ibm_aml', 'amazon_fraud', 'yelp_fraud', 'credit_card_fraud',
             'dgraph_fin', 'ethereum_phishing', 't_finance']:
    d = DATA / name
    if not d.exists():
        print(f"  {name:25s} NAO EXISTE")
        continue
    files = [f for f in d.rglob('*') if f.is_file() and f.name != '.gitkeep']
    total_mb = sum(f.stat().st_size for f in files) / 1e6
    status = f'{len(files)} files ({total_mb:,.0f} MB)' if files else 'VAZIO'
    print(f"  {name:25s} {status}")

In [ ]:
# CELULA 2 - Funcoes utilitarias

@contextlib.contextmanager
def measure_resources():
    proc = psutil.Process(os.getpid())
    m0 = proc.memory_info().rss / 1024**2
    t0 = time.perf_counter()
    r  = {}
    yield r
    r['time_s'] = time.perf_counter() - t0
    r['ram_mb']  = proc.memory_info().rss / 1024**2 - m0


def evaluate_cases_generic(cases, y_edges, n_test, k_frac):
    """Avalia casos para qualquer dataset.
    
    Args:
        cases: lista de dicts com 'nodes', 'seed_edges', 'induced_edges'
        y_edges: array de labels (0/1) para TODAS as arestas usadas como 'full'
        n_test: numero total de arestas de teste
        k_frac: fracao k usada
    """
    y = np.asarray(y_edges, dtype=int)
    pos_total = int(y.sum())
    K = max(1, int(math.ceil(k_frac * n_test)))
    
    if not cases:
        return {m: np.nan for m in [
            'n_cases','coverage','purity_induced','ocr_b100',
            'edges_per_case_median','edges_per_case_max','e_ind_total']}
    
    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    non_empty = [c['induced_edges'] for c in cases if c['induced_edges']]
    if non_empty:
        all_ind = np.unique(np.concatenate(
            [np.asarray(x, dtype=np.int64) for x in non_empty]))
    else:
        all_ind = np.array([], dtype=np.int64)
    
    # Clamp indices
    valid = all_ind[all_ind < len(y)]
    pos_ind = int(y[valid].sum()) if len(valid) > 0 else 0
    coverage = float(pos_ind / max(pos_total, 1))
    purity = float(pos_ind / max(int(ind_sizes.sum()), 1))
    
    return {
        'n_cases': len(cases),
        'coverage': coverage,
        'purity_induced': purity,
        'ocr_b100': float((ind_sizes > 100).mean()),
        'edges_per_case_median': float(np.median(ind_sizes)),
        'edges_per_case_max': float(ind_sizes.max()),
        'e_ind_total': int(ind_sizes.sum()),
    }

print('Funcoes utilitarias definidas.')

In [ ]:
# CELULA 3 - BTCS v3 generico
# Aceita qualquer dataset via interface padronizada:
#   scores: array de probabilidades/scores por aresta
#   src, dst: arrays de endpoints
#   timestamps: array de timestamps (int64)
#   y: labels ground truth (0/1)

def build_Lk(src_sel, dst_sel, ts_sel, delta_L, max_hub_edges=500):
    K = len(src_sel)
    node_map = defaultdict(list)
    for i in range(K):
        node_map[int(src_sel[i])].append((i, int(ts_sel[i])))
        node_map[int(dst_sel[i])].append((i, int(ts_sel[i])))
    lk_edges = set()
    for node, elist in node_map.items():
        if len(elist) < 2:
            continue
        if len(elist) > max_hub_edges:
            elist.sort(key=lambda x: x[1])
            elist = elist[:max_hub_edges]
        else:
            elist.sort(key=lambda x: x[1])
        for i in range(len(elist)):
            ei, ti = elist[i]
            for j in range(i+1, len(elist)):
                ej, tj = elist[j]
                if tj - ti > delta_L:
                    break
                if ei != ej:
                    lk_edges.add((min(ei, ej), max(ei, ej)))
    return list(lk_edges)


def btcs_generic(scores, src, dst, timestamps, y,
                 k=0.01, delta_L=7, resolution=1.0, budget_B=100, seed=42):
    """BTCS v3 generico: funciona com qualquer dataset.
    
    Args:
        scores: array (N,) de scores por aresta (maior = mais suspeito)
        src: array (N,) de source node IDs
        dst: array (N,) de dest node IDs
        timestamps: array (N,) de timestamps (int)
        y: array (N,) de labels ground truth
        k: fracao de top edges
        delta_L: janela temporal para Lk
        resolution: gamma do Leiden
        budget_B: max edges induzidas por caso
        seed: random seed
    
    Returns:
        cases: lista de dicts
    """
    scores = np.asarray(scores, dtype=float)
    src = np.asarray(src, dtype=np.int64)
    dst = np.asarray(dst, dtype=np.int64)
    ts = np.asarray(timestamps, dtype=np.int64)
    N = len(scores)
    K = max(1, int(math.ceil(k * N)))
    sel = np.argsort(-scores)[:K]
    
    src_sel, dst_sel, ts_sel = src[sel], dst[sel], ts[sel]
    max_node = int(max(src.max(), dst.max())) + 1
    print(f'  K={K:,} top edges (of {N:,})')
    
    # Step 1: WCC no subgrafo de nos top-k
    all_nodes = np.unique(np.concatenate([src_sel, dst_sel]))
    nmap = {int(n): i for i, n in enumerate(all_nodes)}
    edges_compact = [(nmap[int(s)], nmap[int(d)])
                     for s, d in zip(src_sel, dst_sel)]
    g_node = ig.Graph(n=len(all_nodes), edges=edges_compact, directed=False)
    g_node.simplify()
    wcc = g_node.connected_components(mode='weak')
    wcc_mem = np.array(wcc.membership, dtype=np.int64)
    n_wcc = int(wcc_mem.max()) + 1
    edge_wcc = np.array([wcc_mem[nmap[int(s)]] for s in src_sel], dtype=np.int64)
    
    # Agrupar por WCC
    wcc_edge_lists = [[] for _ in range(n_wcc)]
    wcc_node_sets = [set() for _ in range(n_wcc)]
    for i in range(K):
        w = int(edge_wcc[i])
        wcc_edge_lists[w].append(i)
        wcc_node_sets[w].update([int(src_sel[i]), int(dst_sel[i])])
    
    # Contar induzidas por WCC (vetorizado)
    wcc_gid = -np.ones(max_node, dtype=np.int64)
    for w, nodes in enumerate(wcc_node_sets):
        for nid in nodes:
            wcc_gid[nid] = w
    g_src_w = np.where(src < max_node, wcc_gid[src], -1)
    g_dst_w = np.where(dst < max_node, wcc_gid[dst], -1)
    mask_w = (g_src_w == g_dst_w) & (g_src_w >= 0)
    idx_w = np.where(mask_w)[0]
    wcc_ind_count = np.zeros(n_wcc, dtype=np.int64)
    if len(idx_w) > 0:
        gw = g_src_w[idx_w]
        uq, cnt = np.unique(gw, return_counts=True)
        wcc_ind_count[uq] = cnt
    
    n_small = int((wcc_ind_count[wcc_ind_count > 0] <= (budget_B or 1e18)).sum())
    n_large = int((wcc_ind_count > (budget_B or 1e18)).sum())
    print(f'  WCC: {n_wcc:,} comps | {n_small} small, {n_large} large')
    
    # Step 2: Hierarquico - split grandes com Leiden
    final_mem = np.full(K, -1, dtype=np.int64)
    next_id = 0
    for w in range(n_wcc):
        comp = wcc_edge_lists[w]
        if not comp:
            continue
        need_split = (budget_B is not None and wcc_ind_count[w] > budget_B
                      and len(comp) >= 2)
        if not need_split:
            for i in comp:
                final_mem[i] = next_id
            next_id += 1
        else:
            comp_arr = np.array(comp, dtype=np.int64)
            lk_local = build_Lk(src_sel[comp_arr], dst_sel[comp_arr],
                                ts_sel[comp_arr], delta_L)
            if not lk_local:
                for i in comp:
                    final_mem[i] = next_id
                    next_id += 1
            else:
                g_local = ig.Graph(n=len(comp), edges=lk_local, directed=False)
                g_local.simplify()
                part = leidenalg.find_partition(
                    g_local, leidenalg.RBConfigurationVertexPartition,
                    resolution_parameter=resolution, seed=seed)
                local_mem = np.array(part.membership, dtype=np.int64)
                n_sub = int(local_mem.max()) + 1
                for j, idx in enumerate(comp):
                    final_mem[idx] = next_id + int(local_mem[j])
                next_id += n_sub
    
    # Step 3: Montar casos (voto majoritario)
    n_total = next_id
    node_votes = defaultdict(lambda: defaultdict(int))
    for i in range(K):
        g = int(final_mem[i])
        if g < 0: continue
        node_votes[int(src_sel[i])][g] += 1
        node_votes[int(dst_sel[i])][g] += 1
    
    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []}
             for _ in range(n_total)]
    for nid, votes in node_votes.items():
        best = max(votes, key=votes.get)
        cases[best]['nodes'].add(nid)
    for i in range(K):
        g = int(final_mem[i])
        if g >= 0:
            cases[g]['seed_edges'].append(int(sel[i]))
    cases = [c for c in cases if c['nodes']]
    
    # Step 4: Arestas induzidas (vetorizado)
    gid_of = -np.ones(max_node, dtype=np.int64)
    for g, case in enumerate(cases):
        for nid in case['nodes']:
            gid_of[nid] = g
    g_src_f = np.where(src < max_node, gid_of[src], -1)
    g_dst_f = np.where(dst < max_node, gid_of[dst], -1)
    mask_f = (g_src_f == g_dst_f) & (g_src_f >= 0)
    idx_f = np.where(mask_f)[0]
    if len(idx_f) > 0:
        gf = g_src_f[idx_f]
        order = np.argsort(gf, kind='stable')
        g_sorted = gf[order]
        i_sorted = idx_f[order]
        uq, cnt = np.unique(g_sorted, return_counts=True)
        for g_id, grp in zip(uq, np.split(i_sorted, np.cumsum(cnt)[:-1])):
            cases[g_id]['induced_edges'] = grp.tolist()
    
    # Step 5: Budget cap
    n_capped = 0
    if budget_B is not None:
        for case in cases:
            if len(case['induced_edges']) > budget_B:
                n_capped += 1
                idx_arr = np.array(case['induced_edges'], dtype=np.int64)
                valid_idx = idx_arr[idx_arr < len(scores)]
                sc_ind = scores[valid_idx]
                top_b = valid_idx[np.argsort(-sc_ind)[:budget_B]]
                case['induced_edges'] = top_b.tolist()
    
    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    print(f'  Final: {len(cases)} cases | capped={n_capped} | '
          f'median={np.median(ind_sizes):.0f} max={ind_sizes.max()}')
    return cases

print('BTCS v3 generico definido.')

In [ ]:
# CELULA 3B - Baselines (mesma interface que btcs_generic)
# Todos recebem (scores, src, dst, timestamps, y, k, budget_B) e retornam cases[]

def baseline_random(scores, src, dst, timestamps, y,
                    k=0.01, budget_B=100, seed=42, **kw):
    """B0: Atribuição aleatória. Top-k edges → grupos aleatórios de ≤B."""
    rng = np.random.RandomState(seed)
    N = len(scores)
    K = max(1, int(math.ceil(k * N)))
    sel = np.argsort(-scores)[:K]
    src_sel, dst_sel = src[sel], dst[sel]
    max_node = int(max(src.max(), dst.max())) + 1

    # Atribuir cada edge a um grupo aleatório
    perm = rng.permutation(K)
    group_size = max(1, budget_B // 2)  # ~B/2 edges por grupo
    n_groups = max(1, K // group_size)
    mem = np.zeros(K, dtype=np.int64)
    for i, idx in enumerate(perm):
        mem[idx] = i % n_groups

    # Montar casos
    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []}
             for _ in range(n_groups)]
    for i in range(K):
        g = int(mem[i])
        cases[g]['nodes'].update([int(src_sel[i]), int(dst_sel[i])])
        cases[g]['seed_edges'].append(int(sel[i]))
    cases = [c for c in cases if c['nodes']]

    # Arestas induzidas + budget cap
    gid_of = -np.ones(max_node, dtype=np.int64)
    for g, case in enumerate(cases):
        for nid in case['nodes']: gid_of[nid] = g
    gs = np.where(src < max_node, gid_of[src], -1)
    gd = np.where(dst < max_node, gid_of[dst], -1)
    mask = (gs == gd) & (gs >= 0)
    idx_f = np.where(mask)[0]
    if len(idx_f) > 0:
        gf = gs[idx_f]
        order = np.argsort(gf, kind='stable')
        uq, cnt = np.unique(gf[order], return_counts=True)
        for g_id, grp in zip(uq, np.split(idx_f[order], np.cumsum(cnt)[:-1])):
            cases[g_id]['induced_edges'] = grp.tolist()
    for case in cases:
        if len(case['induced_edges']) > budget_B:
            idx_arr = np.array(case['induced_edges'], dtype=np.int64)
            valid = idx_arr[idx_arr < len(scores)]
            case['induced_edges'] = valid[np.argsort(-scores[valid])[:budget_B]].tolist()
    print(f'  [B0-Random] {len(cases)} cases')
    return cases


def baseline_wcc_only(scores, src, dst, timestamps, y,
                      k=0.01, budget_B=100, **kw):
    """B1: WCC-only. Top-k edges → WCC, sem Leiden. Budget cap por score."""
    N = len(scores)
    K = max(1, int(math.ceil(k * N)))
    sel = np.argsort(-scores)[:K]
    src_sel, dst_sel = src[sel], dst[sel]
    max_node = int(max(src.max(), dst.max())) + 1

    all_nodes = np.unique(np.concatenate([src_sel, dst_sel]))
    nmap = {int(n): i for i, n in enumerate(all_nodes)}
    edges_compact = [(nmap[int(s)], nmap[int(d)])
                     for s, d in zip(src_sel, dst_sel)]
    g = ig.Graph(n=len(all_nodes), edges=edges_compact, directed=False)
    g.simplify()
    wcc = g.connected_components(mode='weak')
    wcc_mem = np.array(wcc.membership, dtype=np.int64)

    # Cada WCC = 1 caso
    n_wcc = int(wcc_mem.max()) + 1
    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []}
             for _ in range(n_wcc)]
    for i in range(K):
        w = int(wcc_mem[nmap[int(src_sel[i])]])
        cases[w]['nodes'].update([int(src_sel[i]), int(dst_sel[i])])
        cases[w]['seed_edges'].append(int(sel[i]))
    cases = [c for c in cases if c['nodes']]

    # Arestas induzidas + budget cap
    gid_of = -np.ones(max_node, dtype=np.int64)
    for g_id, case in enumerate(cases):
        for nid in case['nodes']: gid_of[nid] = g_id
    gs = np.where(src < max_node, gid_of[src], -1)
    gd = np.where(dst < max_node, gid_of[dst], -1)
    mask = (gs == gd) & (gs >= 0)
    idx_f = np.where(mask)[0]
    if len(idx_f) > 0:
        gf = gs[idx_f]
        order = np.argsort(gf, kind='stable')
        uq, cnt = np.unique(gf[order], return_counts=True)
        for g_id, grp in zip(uq, np.split(idx_f[order], np.cumsum(cnt)[:-1])):
            cases[g_id]['induced_edges'] = grp.tolist()
    for case in cases:
        if len(case['induced_edges']) > budget_B:
            idx_arr = np.array(case['induced_edges'], dtype=np.int64)
            valid = idx_arr[idx_arr < len(scores)]
            case['induced_edges'] = valid[np.argsort(-scores[valid])[:budget_B]].tolist()
    print(f'  [B1-WCC] {len(cases)} cases')
    return cases


def baseline_louvain(scores, src, dst, timestamps, y,
                     k=0.01, budget_B=100, **kw):
    """B2: Louvain community detection no subgrafo top-k.
    Usa igraph community_multilevel (Louvain) no grafo de nós.
    """
    N = len(scores)
    K = max(1, int(math.ceil(k * N)))
    sel = np.argsort(-scores)[:K]
    src_sel, dst_sel = src[sel], dst[sel]
    max_node = int(max(src.max(), dst.max())) + 1

    all_nodes = np.unique(np.concatenate([src_sel, dst_sel]))
    nmap = {int(n): i for i, n in enumerate(all_nodes)}
    edges_compact = [(nmap[int(s)], nmap[int(d)])
                     for s, d in zip(src_sel, dst_sel)]
    g = ig.Graph(n=len(all_nodes), edges=edges_compact, directed=False)
    g.simplify()

    # Louvain
    part = g.community_multilevel()
    mem = np.array(part.membership, dtype=np.int64)
    n_comm = int(mem.max()) + 1

    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []}
             for _ in range(n_comm)]
    for i in range(K):
        c = int(mem[nmap[int(src_sel[i])]])
        cases[c]['nodes'].update([int(src_sel[i]), int(dst_sel[i])])
        cases[c]['seed_edges'].append(int(sel[i]))
    cases = [c for c in cases if c['nodes']]

    # Arestas induzidas + budget cap
    gid_of = -np.ones(max_node, dtype=np.int64)
    for g_id, case in enumerate(cases):
        for nid in case['nodes']: gid_of[nid] = g_id
    gs = np.where(src < max_node, gid_of[src], -1)
    gd = np.where(dst < max_node, gid_of[dst], -1)
    mask = (gs == gd) & (gs >= 0)
    idx_f = np.where(mask)[0]
    if len(idx_f) > 0:
        gf = gs[idx_f]
        order = np.argsort(gf, kind='stable')
        uq, cnt = np.unique(gf[order], return_counts=True)
        for g_id, grp in zip(uq, np.split(idx_f[order], np.cumsum(cnt)[:-1])):
            cases[g_id]['induced_edges'] = grp.tolist()
    for case in cases:
        if len(case['induced_edges']) > budget_B:
            idx_arr = np.array(case['induced_edges'], dtype=np.int64)
            valid = idx_arr[idx_arr < len(scores)]
            case['induced_edges'] = valid[np.argsort(-scores[valid])[:budget_B]].tolist()
    print(f'  [B2-Louvain] {len(cases)} cases')
    return cases


def baseline_greedy_expand(scores, src, dst, timestamps, y,
                           k=0.01, budget_B=100, **kw):
    """B3: Greedy seed expansion.
    Para cada seed edge (top-k por score decrescente), expande BFS
    adicionando vizinhos até atingir budget_B induzidas.
    """
    N = len(scores)
    K = max(1, int(math.ceil(k * N)))
    sel = np.argsort(-scores)[:K]
    src_sel, dst_sel = src[sel], dst[sel]
    max_node = int(max(src.max(), dst.max())) + 1

    # Adjacency list (todo o grafo)
    adj = defaultdict(set)
    edge_of = defaultdict(list)  # node -> list of edge indices
    for idx in range(N):
        s, d = int(src[idx]), int(dst[idx])
        adj[s].add(d)
        adj[d].add(s)
        edge_of[s].append(idx)
        edge_of[d].append(idx)

    assigned = set()  # nodes já atribuidos
    cases = []

    for i in range(K):
        s, d = int(src_sel[i]), int(dst_sel[i])
        if s in assigned and d in assigned:
            continue

        # Iniciar caso com esta seed edge
        case_nodes = {s, d}
        assigned.update(case_nodes)

        # Expandir greedily até budget
        frontier = set()
        for n in case_nodes:
            frontier.update(adj[n] - assigned)

        while frontier:
            # Contar arestas induzidas se adicionarmos cada candidato
            best_node, best_gain = None, -1
            for cand in list(frontier)[:200]:  # limitar busca
                gain = len(adj[cand] & case_nodes)
                if gain > best_gain:
                    best_gain = gain
                    best_node = cand

            if best_node is None:
                break

            # Checar se budget seria violado
            case_nodes.add(best_node)
            # Contar induzidas
            n_ind = 0
            for n in case_nodes:
                for idx in edge_of[n]:
                    s2, d2 = int(src[idx]), int(dst[idx])
                    if s2 in case_nodes and d2 in case_nodes:
                        n_ind += 1
                if n_ind // 2 > budget_B:
                    break
            n_ind = n_ind // 2  # cada aresta contada 2x

            if n_ind > budget_B:
                case_nodes.remove(best_node)
                break

            assigned.add(best_node)
            frontier = set()
            for n in case_nodes:
                frontier.update(adj[n] - assigned)

        # Encontrar arestas induzidas
        induced = []
        for n in case_nodes:
            for idx in edge_of[n]:
                if int(src[idx]) in case_nodes and int(dst[idx]) in case_nodes:
                    induced.append(idx)
        induced = list(set(induced))
        if len(induced) > budget_B:
            idx_arr = np.array(induced, dtype=np.int64)
            induced = idx_arr[np.argsort(-scores[idx_arr])[:budget_B]].tolist()

        cases.append({
            'nodes': case_nodes,
            'seed_edges': [int(sel[i])],
            'induced_edges': induced
        })

        if len(cases) >= K:  # safety
            break

    print(f'  [B3-Greedy] {len(cases)} cases')
    return cases


METHODS = {
    'BTCS_v3': btcs_generic,
    'B0_Random': baseline_random,
    'B1_WCC': baseline_wcc_only,
    'B2_Louvain': baseline_louvain,
    'B3_Greedy': baseline_greedy_expand,
}

print(f'Algoritmos definidos: {", ".join(METHODS.keys())}')

In [ ]:
# CELULA 4 - Loaders por dataset

def load_bitcoin_otc_alpha(name='bitcoin_otc'):
    path = DATA / name
    csvs = list(path.glob('*.csv'))
    assert csvs, f'Nenhum CSV em {path}'
    df = pd.read_csv(csvs[0], header=None,
                     names=['src', 'dst', 'rating', 'timestamp'])
    all_ids = pd.unique(pd.concat([df['src'], df['dst']]))
    id_map = {v: i for i, v in enumerate(sorted(all_ids))}
    src = df['src'].map(id_map).values.astype(np.int64)
    dst = df['dst'].map(id_map).values.astype(np.int64)
    ratings = df['rating'].values.astype(float)
    scores = -ratings
    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
    y = (ratings < 0).astype(int)
    ts_days = (df['timestamp'].values.astype(np.int64) - df['timestamp'].min()) // 86400
    print(f'[{name}] {len(df):,} edges | nodes={len(id_map):,} | '
          f'neg={y.sum():,} ({100*y.mean():.1f}%)')
    return {'name': name, 'scores': scores, 'src': src, 'dst': dst,
            'timestamps': ts_days, 'y': y,
            'n_nodes': len(id_map), 'n_edges': len(df), 'delta_L': 30}


def load_paysim():
    csvs = list((DATA / 'paysim').glob('*.csv'))
    assert csvs, 'PaySim CSV nao encontrado'
    df = pd.read_csv(csvs[0])
    all_ids = pd.unique(pd.concat([df['nameOrig'], df['nameDest']]))
    id_map = {v: i for i, v in enumerate(all_ids)}
    src = df['nameOrig'].map(id_map).values.astype(np.int64)
    dst = df['nameDest'].map(id_map).values.astype(np.int64)
    y = df['isFraud'].values.astype(int)
    timestamps = df['step'].values.astype(np.int64)
    type_risk = df['type'].map({'TRANSFER':0.8,'CASH_OUT':0.9,
        'PAYMENT':0.3,'CASH_IN':0.1,'DEBIT':0.2}).fillna(0.5).values
    amount_norm = (df['amount'].values - df['amount'].min()) / \
                  (df['amount'].max() - df['amount'].min() + 1e-8)
    scores = type_risk * amount_norm
    print(f'[PaySim] {len(df):,} txns | nodes={len(id_map):,} | '
          f'fraud={y.sum():,} ({100*y.mean():.2f}%) [H]')
    return {'name': 'paysim', 'scores': scores, 'src': src, 'dst': dst,
            'timestamps': timestamps, 'y': y,
            'n_nodes': len(id_map), 'n_edges': len(df), 'delta_L': 24}


def load_elliptic():
    path = DATA / 'elliptic'
    feat_csv = list(path.rglob('*features*'))
    edge_csv = list(path.rglob('*edgelist*'))
    class_csv = list(path.rglob('*classes*'))
    assert feat_csv and edge_csv and class_csv, 'Elliptic nao encontrado'
    df_feat = pd.read_csv(feat_csv[0], header=None)
    df_edges = pd.read_csv(edge_csv[0])
    df_class = pd.read_csv(class_csv[0])
    df_class.columns = ['txId', 'class']
    node_label = {}
    for _, row in df_class.iterrows():
        if row['class'] in ['1','2',1,2]:
            node_label[int(row['txId'])] = 1 if str(row['class'])=='1' else 0
    node_ts = dict(zip(df_feat[0].astype(int), df_feat[1].astype(int)))
    df_edges.columns = ['src','dst']
    all_ids = pd.unique(pd.concat([df_edges['src'], df_edges['dst']]))
    id_map = {int(v): i for i, v in enumerate(sorted(all_ids))}
    src = df_edges['src'].map(id_map).values.astype(np.int64)
    dst = df_edges['dst'].map(id_map).values.astype(np.int64)
    y_src = np.array([node_label.get(int(s),0) for s in df_edges['src']])
    y_dst = np.array([node_label.get(int(d),0) for d in df_edges['dst']])
    y = np.maximum(y_src, y_dst)
    ts_src = np.array([node_ts.get(int(s),0) for s in df_edges['src']])
    ts_dst = np.array([node_ts.get(int(d),0) for d in df_edges['dst']])
    timestamps = np.maximum(ts_src, ts_dst).astype(np.int64)
    def ns(nid):
        if nid in node_label: return 1.0 if node_label[nid]==1 else 0.0
        return 0.5
    scores = np.maximum(
        np.array([ns(int(s)) for s in df_edges['src']]),
        np.array([ns(int(d)) for d in df_edges['dst']]))
    scores += np.random.RandomState(42).uniform(0, 0.01, len(scores))
    print(f'[Elliptic] {len(df_edges):,} edges | nodes={len(id_map):,} | '
          f'illicit_edges={y.sum():,} ({100*y.mean():.1f}%) [H]')
    return {'name': 'elliptic', 'scores': scores, 'src': src, 'dst': dst,
            'timestamps': timestamps, 'y': y,
            'n_nodes': len(id_map), 'n_edges': len(df_edges), 'delta_L': 2}


def load_ibm_aml_transactions():
    csv_path = DATA / 'ibm_aml' / 'transactions.csv'
    assert csv_path.exists(), f'{csv_path} nao encontrado'
    df = pd.read_csv(csv_path)
    all_ids = pd.unique(pd.concat([
        df['SENDER_ACCOUNT_ID'].astype(str),
        df['RECEIVER_ACCOUNT_ID'].astype(str)]))
    id_map = {v: i for i, v in enumerate(sorted(all_ids))}
    src = df['SENDER_ACCOUNT_ID'].astype(str).map(id_map).values.astype(np.int64)
    dst = df['RECEIVER_ACCOUNT_ID'].astype(str).map(id_map).values.astype(np.int64)
    y = df['IS_FRAUD'].values.astype(int)
    ts = pd.to_numeric(df['TIMESTAMP'], errors='coerce').fillna(0).values.astype(np.int64)
    if ts.max() > 1e9: ts = (ts - ts.min()) // 86400
    amount = df['TX_AMOUNT'].values.astype(float)
    scores = (amount - amount.min()) / (amount.max() - amount.min() + 1e-8)
    scores += np.random.RandomState(42).uniform(0, 0.01, len(scores))
    print(f'[IBM AML txns] {len(df):,} txns | nodes={len(id_map):,} | '
          f'fraud={y.sum():,} ({100*y.mean():.2f}%) [H]')
    return {'name': 'ibm_aml_txns', 'scores': scores, 'src': src, 'dst': dst,
            'timestamps': ts, 'y': y,
            'n_nodes': len(id_map), 'n_edges': len(df), 'delta_L': 7}


def load_ibm_hili(filename, display_name):
    """Loader genérico para IBM AML HI/LI datasets.
    Todos têm mesmo formato: Timestamp,From Bank,Account,To Bank,Account,...,Is Laundering
    Pandas auto-renomeia col duplicada Account -> Account.1
    """
    csv_path = DATA / 'ibm_aml' / filename
    assert csv_path.exists(), f'{csv_path} nao encontrado'
    df = pd.read_csv(csv_path, header=0)
    # Account = sender, Account.1 = receiver
    src_id = df['From Bank'].astype(str) + '_' + df['Account'].astype(str)
    dst_id = df['To Bank'].astype(str) + '_' + df['Account.1'].astype(str)
    all_ids = pd.unique(pd.concat([src_id, dst_id]))
    id_map = {v: i for i, v in enumerate(all_ids)}
    src = src_id.map(id_map).values.astype(np.int64)
    dst = dst_id.map(id_map).values.astype(np.int64)
    y = df['Is Laundering'].values.astype(int)
    # Timestamp ordinal
    ts_str = df['Timestamp'].astype(str)
    ts_sorted = ts_str.sort_values().unique()
    ts_map = {v: i for i, v in enumerate(ts_sorted)}
    timestamps = ts_str.map(ts_map).values.astype(np.int64)
    # Score: amount normalizado
    amount = df['Amount Paid'].fillna(df['Amount Received']).values.astype(float)
    scores = (amount - amount.min()) / (amount.max() - amount.min() + 1e-8)
    scores += np.random.RandomState(42).uniform(0, 0.001, len(scores))
    print(f'[{display_name}] {len(df):,} txns | nodes={len(id_map):,} | '
          f'laund={y.sum():,} ({100*y.mean():.2f}%) [H]')
    return {'name': display_name, 'scores': scores, 'src': src, 'dst': dst,
            'timestamps': timestamps, 'y': y,
            'n_nodes': len(id_map), 'n_edges': len(df), 'delta_L': 30}


def load_mat_fraud(name, mat_filename, adj_key='homo'):
    import scipy.io as sio
    import scipy.sparse as sp
    mat_path = DATA / name / mat_filename
    assert mat_path.exists(), f'{mat_path} nao encontrado'
    mat = sio.loadmat(str(mat_path))
    adj = mat[adj_key]
    if sp.issparse(adj): adj = adj.tocoo()
    else: adj = sp.coo_matrix(adj)
    labels = np.asarray(mat['label']).flatten()
    n_nodes = len(labels)
    row, col = adj.row, adj.col
    mask_upper = row < col
    src = row[mask_upper].astype(np.int64)
    dst = col[mask_upper].astype(np.int64)
    n_edges = len(src)
    y = np.maximum(labels[src], labels[dst]).astype(int)
    scores = np.maximum(labels[src].astype(float), labels[dst].astype(float))
    scores += np.random.RandomState(42).uniform(0, 0.01, len(scores))
    timestamps = np.arange(n_edges, dtype=np.int64)
    print(f'[{name}] {n_nodes:,} nodes | {n_edges:,} edges | '
          f'fraud_edges={y.sum():,} ({100*y.mean():.1f}%) [H][NT]')
    return {'name': name, 'scores': scores, 'src': src, 'dst': dst,
            'timestamps': timestamps, 'y': y,
            'n_nodes': n_nodes, 'n_edges': n_edges, 'delta_L': n_edges}


def load_aml_pretrained(base, artif_sub, probs_sub, model, seed, name):
    artif = base / artif_sub
    probs = base / probs_sub
    npz = np.load(probs / f'{model}_seed{seed}_test.npz')
    scores = np.asarray(npz['p'], dtype=float)
    y = np.asarray(npz['y'], dtype=int)
    cache = torch.load(artif / 'edge_data_v4_clean.pt',
                       map_location='cpu', weights_only=False)
    ei_all = cache['ei_all_cpu']
    te_idx = cache['te_idx']
    if isinstance(te_idx, torch.Tensor): te_idx = te_idx.numpy()
    if isinstance(ei_all, torch.Tensor): ei_all = ei_all.numpy()
    src = ei_all[0, te_idx].astype(np.int64)
    dst = ei_all[1, te_idx].astype(np.int64)
    candidates = ['transactions.csv','transaction.csv',
                  'hi-large_trans.csv','hi-medium_trans.csv','hi-small_trans.csv',
                  'li-large_trans.csv','li-medium_trans.csv','li-small_trans.csv']
    csv_path = next((list(base.rglob(c))[0] for c in candidates
                     if list(base.rglob(c))), None)
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip().str.lower()
    time_col = next(c for c in ['timestamp','time'] if c in df.columns)
    ts_raw = pd.to_numeric(df[time_col], errors='coerce').fillna(0).astype(np.int64).values
    order = np.argsort(ts_raw, kind='stable')
    ts_sort = ts_raw[order]
    src_col = next(c for c in ['sender_account_id','src'] if c in df.columns)
    dst_col = next(c for c in ['receiver_account_id','dst'] if c in df.columns)
    mask = df[src_col].astype(str).values[order] != df[dst_col].astype(str).values[order]
    timestamps = ts_sort[mask][te_idx]
    print(f'[{name}] {len(scores):,} edges | pos={y.sum():,} ({100*y.mean():.2f}%)')
    return {'name': name, 'scores': scores, 'src': src, 'dst': dst,
            'timestamps': timestamps, 'y': y,
            'n_nodes': int(max(src.max(), dst.max()))+1,
            'n_edges': len(scores), 'delta_L': 7}

print('Loaders definidos.')

In [ ]:
# CELULA 5 - Carregar todos os datasets disponíveis
# Ordem: datasets menores primeiro, depois maiores
# Datasets com scores heuristicos marcados com [H]
# Datasets sem timestamps marcados com [NT]
# IBM HI/LI Large (16 GB cada) só carrega se RAM >= 48 GB

import psutil

datasets = []
RAM_GB = psutil.virtual_memory().total / 1024**3
print(f'RAM total: {RAM_GB:.1f} GB')

# Threshold: Large files (16 GB) precisam de ~48 GB RAM
SKIP_LARGE = RAM_GB < 48
if SKIP_LARGE:
    print('  -> RAM < 48 GB: pulando IBM HI/LI Large variants')

# --- Grupo 1: Datasets com edge-level labels nativos ---

# AML100k/1M (pre-treinados, GNN scores)
try:
    ds = load_aml_pretrained(
        AML100K_BASE, 'artifacts', 'results/probs_v4',
        'SAGE', 42, 'AML100k')
    datasets.append(ds)
except Exception as e:
    print(f'[SKIP] AML100k: {e}')

try:
    ds = load_aml_pretrained(
        AML1M_BASE, 'artifacts',
        'results_aml1m_graphsage_only/probs_v4',
        'GraphSAGE', 44, 'AML1M')
    datasets.append(ds)
except Exception as e:
    print(f'[SKIP] AML1M: {e}')

# Bitcoin OTC / Alpha (edge ratings)
try:
    datasets.append(load_bitcoin_otc_alpha('bitcoin_otc'))
except Exception as e:
    print(f'[SKIP] Bitcoin OTC: {e}')

try:
    datasets.append(load_bitcoin_otc_alpha('bitcoin_alpha'))
except Exception as e:
    print(f'[SKIP] Bitcoin Alpha: {e}')

# IBM AML transactions.csv (1.3M, edge IS_FRAUD) [H]
try:
    datasets.append(load_ibm_aml_transactions())
except Exception as e:
    print(f'[SKIP] IBM AML txns: {e}')

# --- IBM HI variants (High Illicit ratio) ---
try:
    datasets.append(load_ibm_hili('HI-Small_Trans.csv', 'IBM_HI_Small'))
except Exception as e:
    print(f'[SKIP] IBM HI-Small: {e}')

try:
    datasets.append(load_ibm_hili('HI-Medium_Trans.csv', 'IBM_HI_Medium'))
except Exception as e:
    print(f'[SKIP] IBM HI-Medium: {e}')

if not SKIP_LARGE:
    try:
        datasets.append(load_ibm_hili('HI-Large_Trans.csv', 'IBM_HI_Large'))
    except Exception as e:
        print(f'[SKIP] IBM HI-Large: {e}')
else:
    print('[SKIP] IBM HI-Large: RAM insuficiente (16 GB file)')

# --- IBM LI variants (Low Illicit ratio) ---
try:
    datasets.append(load_ibm_hili('LI-Small_Trans.csv', 'IBM_LI_Small'))
except Exception as e:
    print(f'[SKIP] IBM LI-Small: {e}')

try:
    datasets.append(load_ibm_hili('LI-Medium_Trans.csv', 'IBM_LI_Medium'))
except Exception as e:
    print(f'[SKIP] IBM LI-Medium: {e}')

if not SKIP_LARGE:
    try:
        datasets.append(load_ibm_hili('LI-Large_Trans.csv', 'IBM_LI_Large'))
    except Exception as e:
        print(f'[SKIP] IBM LI-Large: {e}')
else:
    print('[SKIP] IBM LI-Large: RAM insuficiente (16 GB file)')

# --- Grupo 2: Datasets com node-level labels (derivar edge) ---

# Elliptic (node -> edge, has timestamps) [H]
try:
    datasets.append(load_elliptic())
except Exception as e:
    print(f'[SKIP] Elliptic: {e}')

# Amazon Fraud (.mat, node-level, no timestamps) [H][NT]
try:
    datasets.append(load_mat_fraud('amazon_fraud', 'Amazon.mat'))
except Exception as e:
    print(f'[SKIP] Amazon Fraud: {e}')

# Yelp Fraud (.mat, node-level, no timestamps) [H][NT]
try:
    datasets.append(load_mat_fraud('yelp_fraud', 'YelpChi.mat'))
except Exception as e:
    print(f'[SKIP] Yelp Fraud: {e}')

# --- Grupo 3: PaySim (~6M txns) [H] ---
try:
    datasets.append(load_paysim())
except Exception as e:
    print(f'[SKIP] PaySim: {e}')

# --- Summary ---
print(f'\n{"="*70}')
print(f'  {len(datasets)} datasets carregados')
print(f'{"="*70}')
for ds in datasets:
    flags = ''
    if ds.get('delta_L', 0) == ds.get('n_edges', -1):
        flags += ' [NT]'
    print(f"  {ds['name']:20s} | {ds['n_nodes']:>10,} nodes | "
          f"{ds['n_edges']:>10,} edges | pos={ds['y'].sum():>8,} "
          f"({100*ds['y'].mean():.1f}%) | dL={ds['delta_L']}{flags}")

In [ ]:
# CELULA 6 - Rodar BTCS + baselines em todos os datasets
# B3_Greedy pula datasets com > 5M edges (adjacency list nao escala)

KS = [0.01, 0.05, 0.10]
GAMMA = 1.0
BUDGET_B = 100

# Threshold para pular B3 (greedy) em datasets enormes
B3_MAX_EDGES = 5_000_000

all_rows = []

for ds in datasets:
    name = ds['name']
    delta_L = ds['delta_L']
    n_edges = ds['n_edges']
    print(f'\n{"="*60}')
    print(f'  {name} ({n_edges:,} edges, delta_L={delta_L})')
    print(f'{"="*60}')

    for method_name, method_fn in METHODS.items():
        # Skip B3_Greedy para datasets grandes (OOM risk)
        if method_name == 'B3_Greedy' and n_edges > B3_MAX_EDGES:
            print(f'\n  [{method_name}] SKIP ({n_edges:,} > {B3_MAX_EDGES:,} edges)')
            for k in KS:
                kp = f'{int(k*100)}%'
                all_rows.append({
                    'dataset': name, 'method': method_name,
                    'n_nodes': ds['n_nodes'], 'n_edges': n_edges,
                    'k%': kp, 'k_frac': k, 'delta_L': delta_L,
                    'resolution': GAMMA, 'budget_B': BUDGET_B,
                    'n_cases': np.nan, 'coverage': np.nan,
                    'purity_induced': np.nan, 'ocr_b100': np.nan,
                    'edges_per_case_median': np.nan, 'edges_per_case_max': np.nan,
                    'e_ind_total': np.nan, 'time_s': np.nan, 'ram_mb': np.nan,
                })
            continue

        for k in KS:
            kp = f'{int(k*100)}%'
            print(f'\n  [{method_name}] k={kp}:')

            try:
                with measure_resources() as res:
                    if method_name == 'BTCS_v3':
                        cases = method_fn(
                            ds['scores'], ds['src'], ds['dst'],
                            ds['timestamps'], ds['y'],
                            k=k, delta_L=delta_L,
                            resolution=GAMMA, budget_B=BUDGET_B)
                    else:
                        cases = method_fn(
                            ds['scores'], ds['src'], ds['dst'],
                            ds['timestamps'], ds['y'],
                            k=k, budget_B=BUDGET_B)
                    m = evaluate_cases_generic(
                        cases, ds['y'], ds['n_edges'], k)

                row = {
                    'dataset': name, 'method': method_name,
                    'n_nodes': ds['n_nodes'], 'n_edges': n_edges,
                    'k%': kp, 'k_frac': k,
                    'delta_L': delta_L,
                    'resolution': GAMMA,
                    'budget_B': BUDGET_B,
                    **m, **res,
                }
                all_rows.append(row)
                print(f"    cov={m['coverage']:.3f} pur={m['purity_induced']:.4f} "
                      f"OCR100={m['ocr_b100']:.3f} #cases={m['n_cases']:,} "
                      f"{res['time_s']:.1f}s")

            except Exception as e:
                print(f'    ERROR: {e}')
                all_rows.append({
                    'dataset': name, 'method': method_name,
                    'n_nodes': ds['n_nodes'], 'n_edges': n_edges,
                    'k%': kp, 'k_frac': k, 'delta_L': delta_L,
                    'resolution': GAMMA, 'budget_B': BUDGET_B,
                    'n_cases': np.nan, 'coverage': np.nan,
                    'purity_induced': np.nan, 'ocr_b100': np.nan,
                    'edges_per_case_median': np.nan, 'edges_per_case_max': np.nan,
                    'e_ind_total': np.nan, 'time_s': np.nan, 'ram_mb': np.nan,
                })

df_multi = pd.DataFrame(all_rows)
print(f'\nTotal: {len(all_rows)} configs = '
      f'{len(datasets)} datasets x {len(METHODS)} methods x {len(KS)} k values')
print(f'(exceto skips por B3_Greedy em datasets grandes)')

In [ ]:
# CELULA 7 - Tabela resumo: BTCS vs Baselines

print('=== Resultados Multi-Dataset: BTCS v3 vs Baselines (B=100) ===\n')

# Tabela principal: coverage por method x dataset (k=1%)
k_focus = '1%'
df_k1 = df_multi[df_multi['k%'] == k_focus].copy()

if len(df_k1) > 0:
    print(f'--- Coverage @ k={k_focus} ---')
    pivot_cov = df_k1.pivot_table(
        index='dataset', columns='method',
        values='coverage', aggfunc='first'
    ).round(4)
    # Reorder columns: BTCS first, then baselines
    col_order = [c for c in ['BTCS_v3','B0_Random','B1_WCC','B2_Louvain','B3_Greedy']
                 if c in pivot_cov.columns]
    pivot_cov = pivot_cov[col_order]
    display(pivot_cov)

    print(f'\n--- Purity @ k={k_focus} ---')
    pivot_pur = df_k1.pivot_table(
        index='dataset', columns='method',
        values='purity_induced', aggfunc='first'
    ).round(4)
    pivot_pur = pivot_pur[[c for c in col_order if c in pivot_pur.columns]]
    display(pivot_pur)

    print(f'\n--- OCR(100) @ k={k_focus} ---')
    pivot_ocr = df_k1.pivot_table(
        index='dataset', columns='method',
        values='ocr_b100', aggfunc='first'
    ).round(4)
    pivot_ocr = pivot_ocr[[c for c in col_order if c in pivot_ocr.columns]]
    display(pivot_ocr)

    print(f'\n--- Time (s) @ k={k_focus} ---')
    pivot_time = df_k1.pivot_table(
        index='dataset', columns='method',
        values='time_s', aggfunc='first'
    ).round(2)
    pivot_time = pivot_time[[c for c in col_order if c in pivot_time.columns]]
    display(pivot_time)

# Full pivot across all k values for BTCS only
print('\n\n--- BTCS v3 across all k values ---')
df_btcs = df_multi[df_multi['method'] == 'BTCS_v3']
pivot_full = df_btcs.pivot_table(
    index='dataset',
    columns='k%',
    values=['coverage', 'purity_induced', 'ocr_b100', 'n_cases'],
    aggfunc='first'
).round(4)
display(pivot_full)

# Dataset summary
print('\n=== Dataset Summary ===')
for ds in datasets:
    fraud_rate = 100 * ds['y'].mean()
    flags = ' [NT]' if ds.get('delta_L', 0) == ds.get('n_edges', -1) else ''
    print(f"  {ds['name']:20s} | {ds['n_nodes']:>10,} nodes | "
          f"{ds['n_edges']:>10,} edges | fraud={fraud_rate:.2f}%{flags}")

# Win/loss: quantas vezes BTCS bate cada baseline
print('\n=== Win Rate (Coverage): BTCS_v3 vs cada baseline ===')
df_valid = df_multi.dropna(subset=['coverage'])
btcs_rows = df_valid[df_valid['method'] == 'BTCS_v3'].set_index(['dataset', 'k%'])
for bl in ['B0_Random', 'B1_WCC', 'B2_Louvain', 'B3_Greedy']:
    bl_rows = df_valid[df_valid['method'] == bl].set_index(['dataset', 'k%'])
    common = btcs_rows.index.intersection(bl_rows.index)
    if len(common) == 0:
        print(f'  vs {bl:12s}: no comparable configs')
        continue
    wins = (btcs_rows.loc[common, 'coverage'].values >=
            bl_rows.loc[common, 'coverage'].values).sum()
    print(f'  vs {bl:12s}: {wins}/{len(common)} wins '
          f'({100*wins/len(common):.0f}%)')

In [ ]:
# CELULA 8 - Plots: BTCS vs Baselines

METHOD_COLORS = {
    'BTCS_v3': '#2ca02c',
    'B0_Random': '#d62728',
    'B1_WCC': '#ff7f0e',
    'B2_Louvain': '#1f77b4',
    'B3_Greedy': '#9467bd',
}

# ---- Plot 1: Coverage comparison (k=1%) por dataset ----
k_focus = '1%'
df_k1 = df_multi[df_multi['k%'] == k_focus].dropna(subset=['coverage'])

if len(df_k1) > 0:
    ds_names = df_k1['dataset'].unique()
    methods = [m for m in ['BTCS_v3','B0_Random','B1_WCC','B2_Louvain','B3_Greedy']
               if m in df_k1['method'].unique()]
    n_methods = len(methods)
    n_ds = len(ds_names)

    fig, ax = plt.subplots(figsize=(max(12, n_ds * 1.2), 6))
    bar_w = 0.8 / n_methods
    x = np.arange(n_ds)

    for j, meth in enumerate(methods):
        vals = []
        for ds_name in ds_names:
            sub = df_k1[(df_k1['dataset'] == ds_name) & (df_k1['method'] == meth)]
            vals.append(sub['coverage'].values[0] if len(sub) > 0 else 0)
        ax.bar(x + j * bar_w, vals, bar_w, label=meth,
               color=METHOD_COLORS.get(meth, '#333'), alpha=0.85)

    ax.set_xticks(x + bar_w * (n_methods - 1) / 2)
    ax.set_xticklabels(ds_names, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Coverage')
    ax.set_title(f'Coverage @ k={k_focus}, B=100 — BTCS v3 vs Baselines')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(OUT / 'btcs_vs_baselines_coverage.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: btcs_vs_baselines_coverage.png')

# ---- Plot 2: Purity comparison (k=1%) ----
if len(df_k1) > 0:
    fig, ax = plt.subplots(figsize=(max(12, n_ds * 1.2), 6))
    for j, meth in enumerate(methods):
        vals = []
        for ds_name in ds_names:
            sub = df_k1[(df_k1['dataset'] == ds_name) & (df_k1['method'] == meth)]
            vals.append(sub['purity_induced'].values[0] if len(sub) > 0 else 0)
        ax.bar(x + j * bar_w, vals, bar_w, label=meth,
               color=METHOD_COLORS.get(meth, '#333'), alpha=0.85)

    ax.set_xticks(x + bar_w * (n_methods - 1) / 2)
    ax.set_xticklabels(ds_names, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Purity (induced)')
    ax.set_title(f'Purity @ k={k_focus}, B=100 — BTCS v3 vs Baselines')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(OUT / 'btcs_vs_baselines_purity.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: btcs_vs_baselines_purity.png')

# ---- Plot 3: BTCS Coverage across k values (all datasets) ----
df_btcs = df_multi[df_multi['method'] == 'BTCS_v3'].dropna(subset=['coverage'])
ds_list = df_btcs['dataset'].unique()
n_ds_btcs = len(ds_list)

if n_ds_btcs > 0:
    n_cols = min(4, n_ds_btcs)
    n_rows = (n_ds_btcs + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5 * n_cols, 4 * n_rows),
                             squeeze=False)

    for idx, ds_name in enumerate(ds_list):
        ax = axes[idx // n_cols][idx % n_cols]
        sub = df_btcs[df_btcs['dataset'] == ds_name].sort_values('k_frac')
        k_vals = sub['k_frac'].values
        covs = sub['coverage'].values
        ocrs = sub['ocr_b100'].values

        ax.bar(range(len(k_vals)), covs, color='#2ca02c', alpha=0.8, label='Cov')
        ax2 = ax.twinx()
        ax2.plot(range(len(k_vals)), ocrs, 'r-o', ms=6, label='OCR')
        ax.set_title(ds_name, fontsize=10)
        ax.set_xticks(range(len(k_vals)))
        ax.set_xticklabels([f'{int(k*100)}%' for k in k_vals])
        ax.set_ylabel('Coverage', color='#2ca02c', fontsize=8)
        ax2.set_ylabel('OCR(100)', color='red', fontsize=8)
        ax.set_ylim(0, 1.05)
        ax2.set_ylim(-0.05, 1.05)

    # Hide empty axes
    for idx in range(n_ds_btcs, n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].set_visible(False)

    plt.suptitle('BTCS v3: Coverage & OCR vs k (B=100)', fontsize=13)
    plt.tight_layout()
    plt.savefig(OUT / 'btcs_coverage_all_datasets.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: btcs_coverage_all_datasets.png')

# ---- Plot 4: Runtime comparison (log scale) ----
df_time = df_multi[df_multi['k%'] == k_focus].dropna(subset=['time_s'])
if len(df_time) > 0:
    fig, ax = plt.subplots(figsize=(max(12, n_ds * 1.2), 6))
    for j, meth in enumerate(methods):
        vals = []
        for ds_name in ds_names:
            sub = df_time[(df_time['dataset'] == ds_name) & (df_time['method'] == meth)]
            vals.append(sub['time_s'].values[0] if len(sub) > 0 else np.nan)
        ax.bar(x + j * bar_w, vals, bar_w, label=meth,
               color=METHOD_COLORS.get(meth, '#333'), alpha=0.85)

    ax.set_xticks(x + bar_w * (n_methods - 1) / 2)
    ax.set_xticklabels(ds_names, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Time (s) — log scale')
    ax.set_yscale('log')
    ax.set_title(f'Runtime @ k={k_focus}, B=100 — BTCS v3 vs Baselines')
    ax.legend(fontsize=8, loc='upper right')
    plt.tight_layout()
    plt.savefig(OUT / 'btcs_vs_baselines_runtime.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: btcs_vs_baselines_runtime.png')

print('\nTodos os plots salvos.')

In [ ]:
# CELULA 9 - Export

df_multi.to_csv(OUT / 'multi_dataset_results.csv', index=False)
print(f'CSV: {OUT}/multi_dataset_results.csv ({len(df_multi)} rows)')

# LaTeX - tabela principal (k=1%, todos os métodos)
cols_tex = ['dataset', 'method', 'k%', 'n_cases', 'coverage',
            'purity_induced', 'ocr_b100', 'time_s']
df_tex = df_multi[cols_tex].round(4)
df_tex.to_latex(OUT / 'multi_dataset_full_table.tex', index=False, escape=False)
print(f'LaTeX (full): {OUT}/multi_dataset_full_table.tex')

# LaTeX - tabela compacta (k=1% only, pivot)
df_k1_tex = df_multi[df_multi['k%'] == '1%'].dropna(subset=['coverage'])
if len(df_k1_tex) > 0:
    pivot_tex = df_k1_tex.pivot_table(
        index='dataset', columns='method',
        values='coverage', aggfunc='first'
    ).round(4)
    col_order = [c for c in ['BTCS_v3','B0_Random','B1_WCC','B2_Louvain','B3_Greedy']
                 if c in pivot_tex.columns]
    pivot_tex = pivot_tex[col_order]
    pivot_tex.to_latex(OUT / 'coverage_pivot_k1.tex', escape=False)
    print(f'LaTeX (pivot): {OUT}/coverage_pivot_k1.tex')

print(f'\nTotal: {len(df_multi)} rows exportados')
print(f'Datasets: {df_multi["dataset"].nunique()}')
print(f'Methods:  {df_multi["method"].nunique()}')

print('\n=== Progresso do Paper ICDM 2026 ===')
print('1.[ok] nb01 - Baselines (B0-B3)')
print('2.[ok] nb02 - BTCS v3 (WCC+Leiden)')
print('3.[ok] nb03 - Ablations (delta_L, B, arch)')
print('4.[ok] nb04 - Multi-Dataset + Baselines (ate 16 datasets x 5 methods)')
print('5.[ok] NP-hardness proof (docs/)')
print('\nProximos passos:')
print('  - Treinar GNNs para datasets com scores heuristicos')
print('  - Refinar delta_L para Amazon/Yelp (sem timestamps)')
print('  - Draft do paper ICDM 2026')